This project investigates what race factors influence the results in Formula 1. We will analyze how starting position influences race outcomes and examine the relative importance of drivers and constructors in shaping performance. We aim to better understand whether success in Formula 1 is driven more by race conditions, driver skill, or team advantages.

In [1]:
import pandas as pd

We will be using those dataset tables: constructors.csv, drivers.csv, results.csv
(Instruction: Write a markdown cell of 1 paragraph describing which dataset tables (among
the multiple options) you will be using. State what each relevant row represents,
how many observations are contained in each table, and a brief overview of the
of the data that is contained there:
▪ Import any necessary libraries.
▪ Import the data.
▪ If necessary, do any calculations for counting the number of rows, etc.
▪ Any code output that you leave in the final notebook will be seen by the
reader of the report. So, it should be formatted nicely as part of the
presentation or removed. )

Merging:
We constructed our main dataset by merging three tables: results, drivers, and constructors. We first selected relevant variables from each dataset(variables that are relavant with race outcomes, driver identity, and constructor information). Before merging, we renamed overlapping columns such as nationality and constructor name to avoid ambiguity in the final dataframe. We then merged the results table with the drivers table using driverId, followed by merging that dataframe with the constructors table using constructorId. For the merging, we use results table as the main dataframe.

In [2]:
df_results = pd.read_csv("data/results.csv")
df_drivers = pd.read_csv("data/drivers.csv")
df_constructors = pd.read_csv("data/constructors.csv")

#check columns
results_col = df_results.columns
drivers_col = df_drivers.columns
constructors_col = df_constructors.columns
print(results_col)
print(drivers_col)
print(constructors_col)




Index(['resultId', 'raceId', 'driverId', 'constructorId', 'number', 'grid',
       'position', 'positionText', 'positionOrder', 'points', 'laps', 'time',
       'milliseconds', 'fastestLap', 'rank', 'fastestLapTime',
       'fastestLapSpeed', 'statusId'],
      dtype='object')
Index(['driverId', 'driverRef', 'number', 'code', 'forename', 'surname', 'dob',
       'nationality', 'url'],
      dtype='object')
Index(['constructorId', 'constructorRef', 'name', 'nationality', 'url'], dtype='object')


In [3]:
#select columns for each dataset
results_sub = df_results[["raceId","driverId", "constructorId", "grid", "positionOrder", "points", "laps"]]
drivers_sub = df_drivers[["driverId", "forename", "surname", "nationality"]]
constructors_sub = df_constructors[["constructorId", "name", "nationality"]]

#rename overlapping columns
drivers_sub = drivers_sub.rename(columns={"nationality": "driver_nationality"})
constructors_sub = constructors_sub.rename(columns={"name": "constructor_name", "nationality": "constructor_nationality"})

#merging
df = pd.merge(left = results_sub, right = drivers_sub, on="driverId", how="left")
df = pd.merge(left = df, right = constructors_sub, on="constructorId", how="left")

Cleaning:
Before the cleaning, we check the key variables used in our analysis(including grid, positionOrder, points, and laps). We found that these variables were already in numeric formats and did not contain any missing values. However, we identified some invalid observations such as observations with a grid position of 0 or a laps of 0. We then filtered them out.

In [4]:
#checking for unusual data in columns with numeric data
df.head()
df.info()
df[["grid", "positionOrder", "points", "laps"]].describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25660 entries, 0 to 25659
Data columns (total 12 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   raceId                   25660 non-null  int64  
 1   driverId                 25660 non-null  int64  
 2   constructorId            25660 non-null  int64  
 3   grid                     25660 non-null  int64  
 4   positionOrder            25660 non-null  int64  
 5   points                   25660 non-null  float64
 6   laps                     25660 non-null  int64  
 7   forename                 25660 non-null  object 
 8   surname                  25660 non-null  object 
 9   driver_nationality       25660 non-null  object 
 10  constructor_name         25660 non-null  object 
 11  constructor_nationality  25660 non-null  object 
dtypes: float64(1), int64(6), object(5)
memory usage: 2.3+ MB


,grid,positionOrder,points,laps
count,25660.000000,25660.000000,25660.000000,25660.000000
mean,11.187256,12.892673,1.854523,45.936204
std,7.251983,7.721729,4.131527,29.867844
min,0.000000,1.000000,0.000000,0.000000
25%,5.000000,6.000000,0.000000,22.000000
50%,11.000000,12.000000,0.000000,52.000000
75%,17.000000,18.000000,2.000000,66.000000
max,34.000000,39.000000,50.000000,200.000000


In [5]:
#remove invalid data
df = df.query("grid > 0 and laps > 0")
#display(df)

Checking after data-preprocessing

In [6]:
#check missing values
print(df[["grid", "positionOrder", "points", "laps"]].isna().sum())

#summary stats
print(df[["grid", "positionOrder", "points", "laps"]].describe().round(2))

grid             0
positionOrder    0
points           0
laps             0
dtype: int64
           grid  positionOrder    points      laps
count  23118.00       23118.00  23118.00  23118.00
mean      11.86          11.52      2.06     50.90
std        6.87           6.65      4.30     27.14
min        1.00           1.00      0.00      1.00
25%        6.00           6.00      0.00     34.00
50%       12.00          11.00      0.00     54.00
75%       17.00          17.00      2.00     68.00
max       34.00          33.00     50.00    200.00
